# ABSim — Tutorial & Testing Notebook

This notebook covers:
1. [Initialising a model](#1-initialising-a-model)
2. [Running a headless simulation and collecting data](#2-headless-simulation)
3. [Exploring scenario toggles](#3-scenario-toggles)
4. [Customising parameters at runtime](#4-customising-parameters)
5. [Launching the interactive GUI](#5-interactive-gui)

**Before running:** make sure the package is installed in the active kernel:
```bash
pip install -e .
```

In [ ]:
import ABSim.config as config
from ABSim.model import Model

---
## 1 — Initialising a model

`Model()` reads its defaults from `config.py`. You can override any toggle or agent count directly in the constructor.

In [ ]:
# Default model — uses values from config.py
model = Model()
model.init_agents('Boid',     config.NUM_BOIDS)
model.init_agents('Predator', config.NUM_PREDS)

print(f"Boids: {len(model.boids)},  Predators: {len(model.predators)}")
print(f"Scenario toggles — social: {model.enable_social}, "
      f"class-diff: {model.enable_classdiff}, predation: {model.enable_predation}")
print(f"Active boid rules: {[r[0] for r in model.active_boid_rules]}")
print(f"Active pred rules: {[r[0] for r in model.active_pred_rules]}")

In [ ]:
# Override counts and toggles explicitly
model2 = Model(n_boids=20, n_preds=2, enable_social=True, enable_classdiff=True, enable_predation=False)
model2.init_agents('Boid',     20)
model2.init_agents('Predator', 2)

print(f"Boids: {len(model2.boids)},  Predators: {len(model2.predators)}")
print(f"Active boid rules: {[r[0] for r in model2.active_boid_rules]}")
print(f"Active pred rules: {[r[0] for r in model2.active_pred_rules]}")

---
## 2 — Headless simulation

Call `model.step()` directly to run without a GUI. The four arguments mirror the live sliders in the GUI:

| Argument | Meaning |
|---|---|
| `avoid_dist` | Minimum distance before boids repel each other |
| `flee_dist` | Distance at which boids detect and flee from predators |
| `catch_dist` | Distance at which a predator catches a boid |
| `speedlim` | Maximum velocity magnitude for all agents |

In [ ]:
N_STEPS  = 300
N_BOIDS  = 100
N_PREDS  = 5

model = Model(n_boids=N_BOIDS, n_preds=N_PREDS,
              enable_social=True, enable_classdiff=True, enable_predation=True)
model.init_agents('Boid',     N_BOIDS)
model.init_agents('Predator', N_PREDS)

boid_counts = [len(model.boids)]
for _ in range(N_STEPS):
    model.step(avoid_dist=10, flee_dist=100, catch_dist=40, speedlim=10)
    boid_counts.append(len(model.boids))

first_eaten = next((i for i, c in enumerate(boid_counts) if c < N_BOIDS), None)
print(f"Start: {boid_counts[0]} boids   End ({N_STEPS} steps): {boid_counts[-1]} boids")
print(f"First boid eaten at step: {first_eaten}")
print(f"Survival rate: {boid_counts[-1] / N_BOIDS:.0%}")

In [ ]:
# Quick plot of boid survival over time — requires matplotlib
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 3))
    plt.plot(boid_counts)
    plt.xlabel("Step")
    plt.ylabel("Boids alive")
    plt.title(f"Boid survival over {N_STEPS} steps ({N_PREDS} predators)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.  Run: pip install matplotlib")

---
## 3 — Scenario toggles

Three boolean toggles in `config.py` control which rules are active. The four documented scenarios:

| ENABLE_SOCIAL | ENABLE_CLASS_DIFF | ENABLE_PREDATION | Description |
|:---:|:---:|:---:|---|
| ✗ | ✗ | ✗ | Random walk — agents wander independently |
| ✓ | ✗ | ✗ | Flocking — all agents share the same social rules |
| ✓ | ✓ | ✗ | Class-differentiated flocking — boids and predators follow different social rules |
| ✓ | ✓ | ✓ | Full model — flocking + hunting + fleeing |

In [ ]:
scenarios = [
    (False, False, False, "Random walk"),
    (True,  False, False, "Flocking (shared rules)"),
    (True,  True,  False, "Class-differentiated flocking"),
    (True,  True,  True,  "Full model"),
]

for s, cd, p, name in scenarios:
    m = Model(enable_social=s, enable_classdiff=cd, enable_predation=p)
    print(f"\n{name}  (S={s}, CD={cd}, P={p})")
    print(f"  Boid rules: {[r[0] for r in m.active_boid_rules]}")
    print(f"  Pred rules: {[r[0] for r in m.active_pred_rules]}")

---
## 4 — Customising parameters

All behavioural constants live in `config.py`. You can override them in-notebook before creating a model, or pass parameters directly to `model.step()`.

In [ ]:
# Compare boid survival with a slow vs fast speed limit
N_STEPS = 200

results = {}
for label, speedlim in [("slow (5)", 5), ("fast (15)", 15)]:
    m = Model(n_boids=100, n_preds=5,
              enable_social=True, enable_classdiff=True, enable_predation=True)
    m.init_agents('Boid', 100)
    m.init_agents('Predator', 5)
    for _ in range(N_STEPS):
        m.step(avoid_dist=10, flee_dist=100, catch_dist=40, speedlim=speedlim)
    results[label] = len(m.boids)

for label, survivors in results.items():
    print(f"{label}: {survivors}/100 boids survived after {N_STEPS} steps")

---
## 5 — Interactive GUI

Running the cell below opens the Tkinter simulation window. **The kernel is blocked until the window is closed.**

You can optionally pass scenario toggles to `run()` to override the `config.py` defaults:
```python
run(enable_social=True, enable_classdiff=True, enable_predation=True)
```

In [ ]:
from ABSim.ui_tk import run
run()